# 原版 TradingAgents 回测

按原版三分析师信号期间，比较 TradingAgents、Buy and Hold、SMA 和 MACD。四种变体的回测入口是 [experiment_analysis.ipynb](experiment_analysis.ipynb)。


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from data import read_prices
from settings import load_config
from backtest import (
    build_baselines, save_baselines, load_baselines, load_original, original_frames,
    select_period, prepare_backtest, compare_positions, plot_backtest,
    monthly_strategy_returns, save_results,
)

ROOT = Path.cwd()


## 参数与实际信号日期

信号按当日收盘对齐。评价窗口从原版信号 CSV 推导，行情历史用于指标预热。

EXCLUDE_PERIODS 控制事后日期筛选。设为 [] 运行完整区间。


In [ ]:
cfg = load_config()
TICKERS = cfg["tickers"]
AGENT_CSV = ROOT / "data/decisions/ta-v0.6.0-deepseek-flash-market-news-fundamentals-main-2026jul-oct-v1.csv"
AGENT_NAME = "TradingAgents (3 analysts)"
RETURN_TYPE = "simple"
NAV_MODE = "additive"  # 保留原净值口径；可改为 compound

# 特殊日期筛选：起止日均包含，可追加多段；设为 [] 恢复完整回测。
# 与区间相交的收盘到下一收盘收益会被排除，所有股票和策略使用同一筛选。
EXCLUDE_PERIODS = [("2026-07-30", "2026-08-03")] #("2026-07-30", "2026-08-03")

# 首个信号日收盘开始评分；完整覆盖由后面的校验检查。
signal_input = pd.read_csv(AGENT_CSV, keep_default_na=False)
signal_dates = sorted(signal_input.information_cutoff.str[:10].unique())
seed_date = signal_dates[0]
backtest_start, backtest_end = signal_dates[0], signal_dates[-1]
periods = {"agent_period": (backtest_start, backtest_end)}
period_label = "Agent signal period"
if EXCLUDE_PERIODS:
    period_label += "\nExcluded: " + ", ".join(f"{a} to {b}" for a, b in EXCLUDE_PERIODS)
period_names = {"agent_period": period_label}
pd.DataFrame(periods, index=["start", "end"]).T


## 1. 读取行情
保留完整历史用于指标预热；收盘到下一收盘收益在合并信号时统一构造。


In [ ]:
# 完整历史用于SMA/MACD预热，收益只在实际信号期间计算。
price_histories = {ticker: read_prices(ticker, cfg, backtest_end) for ticker in TICKERS}
pd.DataFrame([
    {"ticker": ticker, "first_date": prices.date.iloc[0],
     "last_date": prices.date.iloc[-1], "rows": len(prices)}
    for ticker, prices in price_histories.items()
])


## 2. 生成同期规则信号并保存CSV
Buy and Hold、SMA、MACD与Agent共用日期和种子日。先落盘，再读取回测。


In [ ]:
baseline_tables = build_baselines(price_histories, cfg, backtest_start, backtest_end)


In [ ]:
baseline_paths = save_baselines(baseline_tables, ROOT, backtest_start, backtest_end, cfg)
pd.Series(baseline_paths, name="signals_csv")


## 3. 读取原版和规则信号 CSV

每个策略的 CSV 校验交易日和 BUY／SELL／HOLD 动作。

In [ ]:
decision_tables = load_baselines(baseline_paths, backtest_start, backtest_end)


In [ ]:
strategy_names = {
    "buy_hold": "Buy and Hold",
    "sma": f"SMA Trend ({cfg['sma']['fast']}/{cfg['sma']['slow']})",
    "macd": f"MACD Trend ({cfg['macd']['fast']}/{cfg['macd']['slow']}/{cfg['macd']['signal']})",
}

decision_tables.update(load_original(AGENT_CSV, TICKERS, backtest_start, backtest_end))
strategy_names["agent"] = AGENT_NAME
pd.DataFrame([{"ticker": ticker, "strategy": strategy, "strategy_name": strategy_names[strategy],
               "decision_rows": len(rows)} for (ticker, strategy), rows in decision_tables.items()])


## 4. 对齐信号日收盘与收益区间

将原版和规则基线的动作转换为 0 或 1，在信号日收盘建立模拟仓位。各策略从净值 1、仓位 0 开始。

In [ ]:
base_frames = original_frames(price_histories, decision_tables, backtest_start, backtest_end, RETURN_TYPE)
position_tables = {
    (ticker, period, strategy): select_period(frame, start, end)
    for (ticker, strategy), frame in base_frames.items()
    for period, (start, end) in periods.items()
}
pd.concat({key: frame[["close", "signal", "position", "pct"]].head(3)
           for key, frame in position_tables.items()}, names=["ticker", "period", "strategy"])


## 统一回测口径

两本 Notebook 从 backtest.py 导入同一套函数。仓位按信号日收盘对齐，期收益为仓位乘资产收益，扣除相邻仓位差绝对值乘费用。加法净值从 1 开始累计；筛选在计算完整收益和换手之后执行。

## 同期策略比较
按股票比较Agent与三条规则基线的收益、Sharpe、回撤与换手。


In [ ]:
fee = (cfg["commission_bps"] + cfg["slippage_bps"]) / 10000
backtests, metrics_table = compare_positions(
    position_tables, strategy_names, cfg, fee, RETURN_TYPE, NAV_MODE, EXCLUDE_PERIODS)
metrics_table


## 原版净值与回撤

每只股票显示原版策略与三条规则基线。

In [ ]:
cases = {}
for (ticker, period, strategy), frame in backtests.items():
    cases.setdefault((ticker, period), {})[strategy] = frame

figures = {}
for (ticker, period), frames in cases.items():
    metrics = metrics_table.xs((ticker, period), level=("ticker", "period"))
    fig, curves = plot_backtest(frames, strategy_names, metrics,
                               f"{ticker} / original / {period_names[period]}", nav_mode=NAV_MODE)
    figures[ticker, period] = fig
    plt.show()
    plt.close(fig)


## 月收益与结果导出

输出包含指标、月收益、逐期收益、换手、净值及回撤图。run.json 记录实际输入路径和参数。

目录为 results/original/full/ 或 results/original/filtered/。

In [ ]:
monthly_returns = monthly_strategy_returns(backtests, NAV_MODE)
monthly_returns.to_frame()


In [ ]:
mode = "filtered" if EXCLUDE_PERIODS else "full"
output_dir = ROOT / "results" / "original" / mode
record = {
    "experiment": "original", "tickers": sorted(TICKERS), "periods": periods,
    "strategies": strategy_names,
    "agent_csv": str(AGENT_CSV.relative_to(ROOT)),
    "baseline_csvs": {ticker: str(path.relative_to(ROOT)) for ticker, path in baseline_paths.items()},
    "variant_csvs": [],
    "price_basis": "signal_close_to_next_close", "return_type": RETURN_TYPE, "nav_mode": NAV_MODE, "fee": fee, "exclude_periods": EXCLUDE_PERIODS,
}
save_results(output_dir, cases, metrics_table, monthly_returns, figures, record)
str(output_dir)
